In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

In [2]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient

In [3]:
model_client = OpenAIChatCompletionClient(
    model='gpt-4o-mini',
    temperature=0.3
)

In [4]:
from autogen_core.tools import FunctionTool
from langchain_tavily import TavilySearch

In [5]:
tavily_search = TavilySearch()

In [6]:
def search_web(query: str) -> dict:
    """
    Returns web search results for a given query.
    """
    try:
        results = tavily_search.invoke(query)
        return results
    except Exception:
        return {"error": "No results found."}

web_search_tool = FunctionTool(
    search_web,
    description="Performs a web search and returns relevant results for the given query."
)

In [7]:

def percentage_change_fn(start: float, end: float) -> float:
    """
    Returns the percentage change from start to end.
    """
    return ((end - start) / start) * 100

percentage_change_tool = FunctionTool(
    percentage_change_fn,
    description="Returns the percentage change from a starting value to an ending value."
)

In [8]:

planning_agent = AssistantAgent(
    name="PlanningAgent",
    description="An agent for planning tasks, this agent should be the first to engage when given a new task.",
    model_client=model_client,
    system_message="""
    You are a planning agent.
    Your job is to break down complex tasks into smaller, manageable subtasks.
    Your team members are:
        WebSearchAgent: Searches for information
        DataAnalystAgent: Performs calculations

    You only plan and delegate tasks - you do not execute them yourself.

    When assigning tasks, use this format:
    1. <agent> : <task>

    After all tasks are complete, summarize the findings and end with "TERMINATE".
    """,
)

web_search_agent = AssistantAgent(
    name="WebSearchAgent",
    description="An agent for searching information on the web.",
    tools=[web_search_tool],
    model_client=model_client,
    system_message="""
    You are a web search agent.
    Your only tool is search_tool - use it to find information.
    You make only one search call at a time.
    Once you have the results, you never do calculations based on them.
    """,
)

data_analyst_agent = AssistantAgent(
    name="DataAnalystAgent",
    description="An agent for performing calculations.",
    model_client=model_client,
    tools=[percentage_change_tool],
    system_message="""
    You are a data analyst.
    Given the tasks you have been assigned, you should analyze the data and provide results using the tools provided.
    If you have not seen the data, ask for it.
    """,
)

In [9]:
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

In [10]:
text_mention_termination = TextMentionTermination("TERMINATE")
max_messages_termination = MaxMessageTermination(max_messages=25)
termination = text_mention_termination | max_messages_termination

In [11]:
selector_prompt = """Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure the planner agent has assigned tasks before other agents start working.
Only select one agent.
"""

In [12]:
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.ui import Console

In [13]:
team = SelectorGroupChat(
    participants=[planning_agent, web_search_agent, data_analyst_agent],
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True,  
)

In [15]:
task = """
Who was the Chennai Super Kings player with the most runs in the 2018 IPL season, 
and what was the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 seasons?
"""

# Use asyncio.run(...) if you are running this in a script.
await Console(team.run_stream(task=task))

---------- TextMessage (user) ----------

Who was the Chennai Super Kings player with the most runs in the 2018 IPL season, 
and what was the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 seasons?

---------- TextMessage (PlanningAgent) ----------
To address your request, I will break it down into two main subtasks:

1. Find out which Chennai Super Kings player had the most runs in the 2018 IPL season.
2. Calculate the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 IPL seasons.

Now, I will assign the tasks to the appropriate agents:

1. WebSearchAgent : Find the Chennai Super Kings player with the most runs in the 2018 IPL season.
2. WebSearchAgent : Find MS Dhoni's total dismissals for the 2019 and 2020 IPL seasons.
3. DataAnalystAgent : Calculate the percentage change in total dismissals for MS Dhoni between the 2019 and 2020 IPL seasons.

Once the tasks are complete, I will summarize the findings.
---------- ToolCallRequestEv

TaskResult(messages=[TextMessage(id='00fc5c73-4e6a-4c70-867c-d124a242aece', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 23, 14, 9, 5, 440171, tzinfo=datetime.timezone.utc), content="\nWho was the Chennai Super Kings player with the most runs in the 2018 IPL season, \nand what was the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 seasons?\n", type='TextMessage'), TextMessage(id='75415fb2-7eb2-4605-aecf-4dc1b3ed751c', source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=1719, completion_tokens=168), metadata={}, created_at=datetime.datetime(2025, 7, 23, 14, 9, 10, 460004, tzinfo=datetime.timezone.utc), content="To address your request, I will break it down into two main subtasks:\n\n1. Find out which Chennai Super Kings player had the most runs in the 2018 IPL season.\n2. Calculate the percentage change in MS Dhoni's total dismissals between the 2019 and 2020 IPL seasons.\n\nNow, I will assign the tasks to t